# 문제 3-1 : 콘텐츠분쟁해결 RAG 시스템

콘텐츠분쟁해결 사례집(`data/콘텐츠분쟁해결_사례.pdf`)을 활용한 **법률 자문 RAG 시스템**을 실습 가이드의 단계 순서대로 구축합니다.

| 단계 | 내용 |
|---|---|
| 0 | 문서 로드 (`PyPDFLoader`) |
| 1 | 문서 분할 (`RecursiveCharacterTextSplitter`, chunk 1500 / overlap 300, 【사건개요】 등 섹션 구분자) |
| 2 | 임베딩 (`UpstageEmbeddings` solar-embedding-1-large) + FAISS 벡터스토어 |
| 3 | 검색기 (similarity, k=5) |
| 4 | LLM (`ChatUpstage` solar-pro3, temperature 0.2) |
| 5 | 법률 자문 프롬프트 |
| 6 | LCEL 체인 |
| 7 | 테스트 질문 5개 |
| 8 | 분쟁 유형 분류 함수 (선택) |

### 라이브러리 설치 & API Key 확인

In [ ]:
# uv add pypdf langchain-community langchain-text-splitters langchain-upstage faiss-cpu python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# ChatUpstage, UpstageEmbeddings 가 UPSTAGE_API_KEY 환경 변수를 자동으로 읽음 (키 값은 출력하지 않음)
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
print("UPSTAGE_API_KEY 설정 확인 완료")

UPSTAGE_API_KEY 설정 확인 완료


## 0단계: 문서 로드
목차 페이지(점선 `····`)와 빈 페이지는 검색 결과 자리를 차지하므로 제외합니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

print("==> 0. 문서 로드 → PDF 읽기...")
loader = PyPDFLoader('../data/콘텐츠분쟁해결_사례.pdf')
pages = loader.load()
print(f"  총 {len(pages)}페이지 로드 완료")

# 빈 페이지(50자 미만)와 목차 페이지(점선이 많은 페이지) 제외
documents = [
    doc for doc in pages
    if len(doc.page_content.strip()) >= 50 and doc.page_content.count("····") < 5
]
print(f"  빈 페이지/목차 제외 후 {len(documents)}페이지 사용")

==> 0. 문서 로드 → PDF 읽기...
  총 109페이지 로드 완료
  빈 페이지/목차 제외 후 95페이지 사용


## 1단계: 문서 분할 설정

> **전처리 이유**: `PyPDFLoader` 로 읽은 텍스트는 가이드의 구분자 `"\n【사건개요】"` 와 글자가 정확히 일치하지 않습니다.
> - pypdf 4.x(수업 프로젝트 버전): `【사건개요 】` 처럼 **괄호 안에 공백**이 들어감 → 일치 0회
> - pypdf 5.x: `...요구【사건개요】신청인은...` 처럼 **앞에 줄바꿈이 없음** → 일치 0회
>
> 그대로 두면 섹션 구분자가 하나도 동작하지 않으므로, 분할 전에 섹션 표시를 `"\n【사건개요】"` 형태로 통일합니다.

In [ ]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 섹션 표시(【사건개요】【쟁점사항】【처리경위】【처리결과】) 앞에 줄바꿈 추가 → 가이드의 구분자가 동작하도록
# pypdf 버전에 따라 "【사건개요 】"처럼 괄호 안에 공백이 들어가므로 공백을 허용해 찾고 "\n【사건개요】" 형태로 통일
SECTION_PATTERN = re.compile(r"\s*【\s*(사건개요|쟁점사항|처리경위|처리결과)\s*】")
for doc in documents:
    doc.page_content = SECTION_PATTERN.sub(r"\n【\1】", doc.page_content)

# 전처리 확인: 가이드 구분자가 실제로 몇 번 나타나는지
all_text = "\n".join(doc.page_content for doc in documents)
for sep in ["\n【사건개요】", "\n【쟁점사항】", "\n【처리경위】", "\n【처리결과】"]:
    print(f"  {sep!r}: {all_text.count(sep)}회")

print("==> 1. 문서 분할 → 작은 청크로 나누기")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,        # 법률 사례는 1200-1800자 권장
    chunk_overlap=300,      # 사례 맥락 보존을 위해 200-400자
    separators=[
        "\n【사건개요】",     # 법률 문서 섹션 구분자
        "\n【쟁점사항】",     # 쟁점 부분 구분
        "\n【처리경위】",     # 처리 과정 구분
        "\n【처리결과】",     # 결과 부분 구분
        "\n■", "\n\n", "\n", ".", " ", ""
    ]
)

chunks = text_splitter.split_documents(documents)
print(f"  {len(chunks)}개 청크 생성 완료")
print(f"  평균 청크 길이: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f}자")
print("\n[청크 예시]\n", chunks[3].page_content[:400])

  '\n【사건개요】': 56회
  '\n【쟁점사항】': 51회
  '\n【처리경위】': 56회
  '\n【처리결과】': 51회
==> 1. 문서 분할 → 작은 청크로 나누기
  95개 청크 생성 완료
  평균 청크 길이: 764자

[청크 예시]
 콘텐츠분쟁조정 법리 연구 2부 - 타 분쟁조정사례 조사 -
6에게 반환한 것을 부당하다고 보기 어렵다 .
【처리결과】
신청인에게 상기사항을 안내함 .


## 2단계: 임베딩 모델 설정 + 벡터스토어(FAISS)
처음 실행할 때만 임베딩 API 를 호출해 인덱스를 만들고, 다음부터는 저장된 인덱스를 불러옵니다.
(분할 설정을 바꾸면 `DB_PATH` 폴더를 삭제하고 다시 실행하세요.)

In [ ]:
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings

# FAISS 인덱스 저장 경로 (경로 구분자는 슬래시 사용)
DB_PATH = "../db/faiss_content_dispute_upstage"

print("==> 2. 벡터화 → 임베딩으로 변환")
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

if Path(DB_PATH).exists():
    # 직접 만든 인덱스이므로 역직렬화 허용
    vectorstore = FAISS.load_local(DB_PATH, embeddings, allow_dangerous_deserialization=True)
    print(f"  기존 인덱스 로드: {DB_PATH} ({vectorstore.index.ntotal}개 벡터)")
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(DB_PATH)
    print(f"  인덱스 생성 및 저장: {DB_PATH} ({len(chunks)}개 벡터)")

==> 2. 벡터화 → 임베딩으로 변환
  인덱스 생성 및 저장: ../db/faiss_content_dispute_upstage (95개 벡터)


## 3단계: 검색기 설정

In [ ]:
print("==> 3. 검색 → 질문과 유사한 사례 찾기")
retriever = vectorstore.as_retriever(
    search_type="similarity",        # 또는 "mmr" (다양성 고려시)
    search_kwargs={"k": 5}           # 상위 5개 관련 사례 검색
)

# 검색 테스트
for doc in retriever.invoke("게임 아이템이 시스템 오류로 사라졌어요"):
    print(f"  P{doc.metadata.get('page_label', '?')}: {doc.page_content[:70].replace(chr(10), ' ')}...")

==> 3. 검색 → 질문과 유사한 사례 찾기
  P9: 1. 게임 3게임1 ■ 1-가-1) 한국소비자원 조정전 상담사례 2006_시스템 오류로 소멸된 아이템 복구 요구 【사건개요】 ...
  P29: 1. 게임 23■ 1-가-1) 한국소비자원 조정전 상담사례 2009_시스템 오류로 인한 손실 아이템 복구 요구 【사건개요】 신...
  P10: 콘텐츠분쟁조정 법리 연구 2부 - 타 분쟁조정사례 조사 - 4■ 1-가-1) 한국소비자원 조정전 상담사례 2006_프로그램 오...
  P11: 1. 게임 5■ 1-가-1) 한국소비자원 조정전 상담사례 2007_인터넷게임서비스 아이템 복구 요구 【사건개요】 신청인은 피신...
  P31: 1. 게임 25■ 1-가-1) 한국소비자원 조정전 상담사례 2009_해킹으로 손실된 아이템 복구 요청 【사건개요】 피청구인이 ...


## 4단계: LLM 설정

In [ ]:
from langchain_upstage import ChatUpstage

print("==> 4. 생성 → LLM 설정")
llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.2
    )
print(llm.model_name)

==> 4. 생성 → LLM 설정
solar-pro3


## 5단계: 법률 자문 프롬프트 작성

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다.
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요
2. 관련 법령이나 조항이 있다면 명시하세요
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요
4. 실무적 해결방안을 단계별로 제시하세요
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요

전문 법률 조언:"""

prompt = PromptTemplate.from_template(prompt_template)
print(prompt.input_variables)

['context', 'question']


## 6단계: LCEL 체인 생성
가이드의 `qa_chain` 과 함께, 참조한 사례(페이지)를 보여주기 위해 **검색 1회로 답변과 출처를 함께 반환**하는 `rag_with_sources()` 도 만듭니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    """검색된 문서들을 하나의 문자열로 포맷"""
    return "\n\n".join(doc.page_content for doc in docs)


qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# 소스 문서 포함 버전 (검색을 한 번만 하고, 같은 검색 결과로 답변 + 출처 반환)
def rag_with_sources(question):
    docs = retriever.invoke(question)
    answer = (prompt | llm | StrOutputParser()).invoke({"context": format_docs(docs), "question": question})
    return {"answer": answer, "source_documents": docs}


print("  LCEL RAG 파이프라인 구축 완료!")

  LCEL RAG 파이프라인 구축 완료!


## 7단계: 테스트 질문 작성

In [ ]:
test_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?"
]

print("\n" + "=" * 70)
print(" 콘텐츠분쟁해결 RAG 시스템 테스트")
print("=" * 70)

for i, question in enumerate(test_questions, 1):
    print(f"\n【테스트 {i}/{len(test_questions)}】")
    print(f"Q: {question}\n")

    result = rag_with_sources(question)
    print(f"A: {result['answer']}\n")

    # 참조 사례 (page_label: 문서에 표시된 페이지 번호)
    docs = result["source_documents"]
    print(f"  참조: {len(docs)}개 사례")
    for j, doc in enumerate(docs[:3], 1):
        print(f"    {j}. P{doc.metadata.get('page_label', '?')}: {doc.page_content[:80].replace(chr(10), ' ')}...")
    print("-" * 70)


 콘텐츠분쟁해결 RAG 시스템 테스트

【테스트 1/5】
Q: 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?

A: **온라인 게임 시스템 오류로 인한 아이템 소멸 시 복구 요청 관련 법률 조언**

---

### 1. **쟁점 요약**
- **신청인의 주장**: 시스템 오류로 인해 게임 내 아이템이 소멸되었으며, 이에 대한 원상복구 또는 보상이 필요하다.
- **피신청인의 입장**: 약관상 계정 명의자 외 복구 불가, 시스템 오류 발생 여부 불확실, 보안 미준수 등으로 복구 거부.

---

### 2. **관련 법령 및 약관 조항**

#### 가. **민법**
- **제250조(도품, 유실물에 대한 특례)**  
  > 도품 또는 유실물은 피해자 또는 유실자가 도난 또는 유실한 날로부터 2년 내에 반환을 청구할 수 있다.  
  > 단, 금전인 경우에는 적용되지 않는다.

- **제251조(양수인의 선의)**  
  > 도품 또는 유실물을 선의로 매수한 경우, 피해자는 대가를 변상하고 반환을 청구할 수 있다.

#### 나. **게임사 이용약관**
- **계정 소유권 및 거래 제한 조항**  
  > 대부분의 게임사는 계정 명의자 외 복구 요청을 거절하며, 현금 거래 및 계정 공유를 금지한다.
- **보안 정책**  
  > 해킹 피해 발생 시 보안 서비스(예: OTP, 보안카드 등) 사용 여부에 따라 보상 여부가 결정된다.

---

### 3. **유사 사례 분석 및 처리 경위**

#### 가. **2006년 사례 (시스템 오류로 인한 아이템 소멸)**
- **쟁점**: 시스템 오류로 인한 아이템 소멸 여부
- **처리 경위**:  
  - 피신청인은 계정 명의자가 아닌 경우 복구 불가라고 주장.
  - 신청인은 계정 명의자가 타인임을 인정하나, 실소유주임을 주장.
  - 약관상 계정 공유 및 현금 거래 금지로 인해 복구 불가.
- **결과**: 계정 명의자가 직접 복구 신청 

## 8단계: 분쟁 유형 분류 함수 (선택)
질문의 키워드로 분쟁 유형을 분류합니다. 위에서부터 순서대로 검사하므로 여러 유형의 키워드가 함께 있으면 먼저 검사한 유형(게임)이 선택됩니다.

In [ ]:
def classify_dispute_type(query):
    game_keywords = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드", "온라인게임"]
    elearning_keywords = ["강의", "온라인교육", "이러닝", "수강", "환불", "화상교육"]
    web_keywords = ["웹사이트", "무료체험", "자동결제", "구독", "사이트"]

    query_lower = query.lower()

    if any(keyword in query_lower for keyword in game_keywords):
        return "게임"
    elif any(keyword in query_lower for keyword in elearning_keywords):
        return "이러닝"
    elif any(keyword in query_lower for keyword in web_keywords):
        return "웹콘텐츠"
    else:
        return "기타"


# 테스트 질문 분류 결과 (LLM 호출 없음)
for q in test_questions:
    print(f"[{classify_dispute_type(q)}] {q}")

[게임] 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?
[이러닝] 인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?
[이러닝] 무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?
[게임] 미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?
[기타] 온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?


#### 분류 결과를 RAG 에 활용
분류한 유형을 질문 앞에 붙여(`[게임 분쟁] ...`) 같은 분야의 사례가 검색되도록 돕습니다. `기타` 는 질문을 그대로 사용합니다.

In [ ]:
def ask_with_type(question):
    dispute_type = classify_dispute_type(question)
    print(f"▶ 분쟁 유형: {dispute_type}")
    query = question if dispute_type == "기타" else f"[{dispute_type} 분쟁] {question}"
    result = rag_with_sources(query)
    print(f"\nQ: {question}\n\nA: {result['answer']}")
    return result


ask_with_type("게임 계정이 해킹당해서 아이템을 모두 잃어버렸어요. 보상받을 수 있나요?")

▶ 분쟁 유형: 게임

Q: 게임 계정이 해킹당해서 아이템을 모두 잃어버렸어요. 보상받을 수 있나요?

A: **전문 법률 조언: 게임 계정 해킹으로 인한 아이템 손실 보상 여부**

---

### 1. **제시된 사례 요약 및 법적 쟁점 정리**

#### **사례 1: 2009년 해킹으로 인한 아이템 손실 (한국소비자원 조정전 상담사례)**
- **사건 개요**: 게임 접속 후 4시간 뒤 계정 해킹 확인, 모든 아이템(8만원 상당) 소실.
- **쟁점**: 해킹으로 인한 아이템 복구 여부.
- **처리 경위**: 게임사는 보안 프로그램 2개 이상 사용 의무 규정 적용, 신청인은 1개만 사용 → 보상 불가.
- **처리 결과**: 복구 불가 안내.

#### **사례 2: 2008년 해킹으로 인한 아이템 소실 (한국소비자원 조정전 상담사례)**
- **사건 개요**: 해킹 발생 후 40분 내 아이템 거래, 대부분 제3자에게 이동.
- **쟁점**: 해킹으로 소실된 6종 아이템 복구 여부.
- **처리 경위**: 해킹 신고 지연 인정 → 6종 중 4종 복구, 나머지 2종은 거래 완료로 복구 불가.
- **처리 결과**: 신청인 수용.

#### **사례 3: 2007년 아이템 회수 사건 (한국소비자원 조정결정례)**
- **사건 개요**: 타 이용자가 해킹으로 분실한 아이템을 신청인이 구매 → 게임사가 회수.
- **쟁점**: 게임사의 아이템 회수 행위의 정당성 여부.
- **처리 경위**: 아이템은 해킹으로 분실된 것으로, 신청인은 게임머니 지급 후 환급받음 → 피해 없음.
- **처리 결과**: 회수 정당 인정.

#### **사례 4: 2007년 계정 영구 이용제한 사건 (한국소비자원 조정결정례)**
- **사건 개요**: 아이템 현금거래 후 계정 영구 이용제한 조치.
- **쟁점**: 계정 영구 이용제한 조치가 과도한 규제인지 여부.
- **처리 경위**: 공정거래위원회는 ‘처음 적발 시 곧바로 영구 제한’ 조항은 불공정 약관으로 판단.
- **처리 

{'answer': '**전문 법률 조언: 게임 계정 해킹으로 인한 아이템 손실 보상 여부**\n\n---\n\n### 1. **제시된 사례 요약 및 법적 쟁점 정리**\n\n#### **사례 1: 2009년 해킹으로 인한 아이템 손실 (한국소비자원 조정전 상담사례)**\n- **사건 개요**: 게임 접속 후 4시간 뒤 계정 해킹 확인, 모든 아이템(8만원 상당) 소실.\n- **쟁점**: 해킹으로 인한 아이템 복구 여부.\n- **처리 경위**: 게임사는 보안 프로그램 2개 이상 사용 의무 규정 적용, 신청인은 1개만 사용 → 보상 불가.\n- **처리 결과**: 복구 불가 안내.\n\n#### **사례 2: 2008년 해킹으로 인한 아이템 소실 (한국소비자원 조정전 상담사례)**\n- **사건 개요**: 해킹 발생 후 40분 내 아이템 거래, 대부분 제3자에게 이동.\n- **쟁점**: 해킹으로 소실된 6종 아이템 복구 여부.\n- **처리 경위**: 해킹 신고 지연 인정 → 6종 중 4종 복구, 나머지 2종은 거래 완료로 복구 불가.\n- **처리 결과**: 신청인 수용.\n\n#### **사례 3: 2007년 아이템 회수 사건 (한국소비자원 조정결정례)**\n- **사건 개요**: 타 이용자가 해킹으로 분실한 아이템을 신청인이 구매 → 게임사가 회수.\n- **쟁점**: 게임사의 아이템 회수 행위의 정당성 여부.\n- **처리 경위**: 아이템은 해킹으로 분실된 것으로, 신청인은 게임머니 지급 후 환급받음 → 피해 없음.\n- **처리 결과**: 회수 정당 인정.\n\n#### **사례 4: 2007년 계정 영구 이용제한 사건 (한국소비자원 조정결정례)**\n- **사건 개요**: 아이템 현금거래 후 계정 영구 이용제한 조치.\n- **쟁점**: 계정 영구 이용제한 조치가 과도한 규제인지 여부.\n- **처리 경위**: 공정거래위원회는 ‘처음 적발 시 곧바로 영구 제한’ 조항은 불공정 약관으로 판단.\n- **처리 결과**: 계정 영구 이용제한 조치 취

## (참고) 프롬프트 기본 템플릿과 비교
가이드의 **기본 법률 자문 프롬프트**(간단 버전)로 같은 질문에 답변하고, 5단계의 상세 프롬프트와 비교합니다.

In [ ]:
# 기본 법률 자문 프롬프트 (5분 만에 완성)
basic_prompt_template = """당신은 콘텐츠 분야 전문 법률 자문사입니다.

관련 분쟁사례: {context}

상담 내용: {question}

답변 가이드라인:
1. 사례를 근거로 답변하세요.
2. 관련 법령을 명시하세요.
3. 단계별 해결방안을 제시하세요.
4. 유사 사례를 참조하세요.
5. 없는 정보는 "확인할 수 없습니다"라고 하세요."""

basic_prompt = PromptTemplate.from_template(basic_prompt_template)

basic_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | basic_prompt
    | llm
    | StrOutputParser()
)

question = test_questions[0]
print(f"Q: {question}\n")
print("[기본 프롬프트]\n", basic_chain.invoke(question))
print("\n" + "-" * 70)
print("[상세 프롬프트 (5단계)]\n", qa_chain.invoke(question))

Q: 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?

[기본 프롬프트]
 **온라인 게임에서 시스템 오류로 아이템이 사라졌을 때 게임회사의 복구 거부에 대한 해결 방안**

---

### 1. **관련 사례 및 법리 분석**

#### **사례 1: 2006년 시스템 오류로 소멸된 아이템 복구 요구**
- **사건개요**: 시스템 오류로 120,000원 상당의 아이템이 소멸되었으나, 계정 명의자가 아니라는 이유로 복구가 거부됨.
- **처리결과**: 계정 명의자가 직접 복구 신청을 해야 하며, 명의자가 아닌 경우 처리가 불가함.
- **법리**: 게임사의 약관에서 **계정 공유 및 현금 거래를 금지**하고 있으며, 계정 명의자만이 권리를 주장할 수 있음.

#### **사례 2: 2009년 시스템 오류로 인한 게임머니 소실**
- **사건개요**: 프로그램 업데이트 후 렉(시스템 오류)으로 인해 게임머니 1,000억 소실 주장.
- **처리결과**: 시스템 오류 발생 증거가 없으며, 신청인의 통신 회선 또는 PC 오류로 판단됨.
- **법리**: 시스템 오류 발생 여부는 **객관적 증거**가 필요하며, 단순 사용자 주장만으로는 복구가 어려움.

#### **사례 3: 2006년 프로그램 오류로 아이템 소멸**
- **사건개요**: 프로그램 오류로 아이템이 사라졌으나, 게시판 신청 후 1주일 동안 답변이 없음.
- **처리결과**: 프로그램 오류로 인한 아이템 소멸이 인정되어 복구 완료.
- **법리**: 게임사가 **프로그램 오류를 인정**하고 복구 절차를 진행하면 복구가 가능함.

#### **사례 4: 2007년 해킹으로 인한 아이템 회수**
- **사건개요**: 타 이용자가 해킹으로 분실한 아이템을 회수하고, 구매자에게 게임머니 환급.
- **처리결과**: 아이템은 소유자에게 반환되었으며, 구매자에게 피해가 없음.
- **법리**: **민법 제250조**에 따라 도품·유실물은 2년 내